#Derin Öğrenme Dersi Ödev 2

Mehmet Oğuz Gedik

24120205062

Bilgisayar Mühendisliği 3. sınıf

##1- Donanım kontrolü ve Kütüphaneleri hazırlama

In [1]:
!pip install -q ultralytics gradio

import torch
import ultralytics

print("PyTorch Sürümü:", torch.__version__)
print("CUDA Aktif mi?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Modeli:", torch.cuda.get_device_name(0))
ultralytics.checks()

Ultralytics 8.4.212 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 0.1/235.7 GB disk)


##2-Veri Setinin İndirilmesi ve Train, Val ve Test Ayrımının Doğrulanması

In [2]:
import os
import shutil
import random
from pathlib import Path
from ultralytics.utils.downloads import download

from ultralytics.data.utils import check_det_dataset
data_info = check_det_dataset("VOC.yaml")

voc_root = Path(data_info["path"])
print("Veri seti ana dizini:", voc_root)

test_img_dir = voc_root / "images" / "test2007"
test_lbl_dir = voc_root / "labels" / "test2007"

val_split_img = voc_root / "images" / "val_custom"
val_split_lbl = voc_root / "labels" / "val_custom"
test_split_img = voc_root / "images" / "test_custom"
test_split_lbl = voc_root / "labels" / "test_custom"

for d in [val_split_img, val_split_lbl, test_split_img, test_split_lbl]:
    d.mkdir(parents=True, exist_ok=True)

all_test_images = sorted(list(test_img_dir.glob("*.jpg")))
random.seed(42)
random.shuffle(all_test_images)

mid = len(all_test_images) // 2
val_files = all_test_images[:mid]
test_files = all_test_images[mid:]

for img_path in val_files:
    shutil.copy(img_path, val_split_img / img_path.name)
    lbl_path = test_lbl_dir / f"{img_path.stem}.txt"
    if lbl_path.exists():
        shutil.copy(lbl_path, val_split_lbl / lbl_path.name)

for img_path in test_files:
    shutil.copy(img_path, test_split_img / img_path.name)
    lbl_path = test_lbl_dir / f"{img_path.stem}.txt"
    if lbl_path.exists():
        shutil.copy(lbl_path, test_split_lbl / lbl_path.name)

custom_yaml = f"""
path: {voc_root}
train:
  - images/train2012
  - images/train2007
  - images/val2012
  - images/val2007
val: images/val_custom
test: images/test_custom

names:
  0: aeroplane
  1: bicycle
  2: bird
  3: boat
  4: bottle
  5: bus
  6: car
  7: cat
  8: chair
  9: cow
  10: diningtable
  11: dog
  12: horse
  13: motorbike
  14: person
  15: pottedplant
  16: sheep
  17: sofa
  18: train
  19: tvmonitor
"""

with open("voc_custom.yaml", "w") as f:
    f.write(custom_yaml.strip())

train_count = sum(len(list((voc_root / "images" / p).glob("*.jpg"))) for p in ["train2012", "train2007", "val2012", "val2007"])
print(f"Eğitim (Train) görüntü sayısı: {train_count}")
print(f"Doğrulama (Val) görüntü sayısı: {len(val_files)}")
print(f"Test görüntü sayısı: {len(test_files)}")


WARNING ⚠️ Dataset 'VOC.yaml' images not found, missing path '/content/datasets/VOC/images/test2007'
Unzipping /content/datasets/VOC/images/VOCtrainval_06-Nov-2007.zip to /content/datasets/VOC/images/VOCdevkit...: 100% ━━━━━━━━━━━━ 10945/10945 454.2files/s 24.1s
Unzipping /content/datasets/VOC/images/VOCtest_06-Nov-2007.zip to /content/datasets/VOC/images/VOCdevkit...: 100% ━━━━━━━━━━━━ 10357/10357 445.0files/s 23.3s
Unzipping /content/datasets/VOC/images/VOCtrainval_11-May-2012.zip to /content/datasets/VOC/images/VOCdevkit...: 100% ━━━━━━━━━━━━ 40189/40189 1.4Kfiles/s 28.1s
train2012: 100% ━━━━━━━━━━━━ 5717/5717 2.9Kit/s 2.0s
val2012: 100% ━━━━━━━━━━━━ 5823/5823 3.1Kit/s 1.9s
train2007: 100% ━━━━━━━━━━━━ 2501/2501 2.1Kit/s 1.2s
val2007: 100% ━━━━━━━━━━━━ 2510/2510 2.4Kit/s 1.0s
test2007: 100% ━━━━━━━━━━━━ 4952/4952 2.5Kit/s 2.0s
Dataset download success ✅ (64.3s), saved to /content/datasets

Veri seti ana dizini: /content/datasets/VOC
Eğitim (Train) görüntü sayısı: 16551
Doğrulama (V

##3-Model Eğitimi ve Performans Değerlendirmesi

In [3]:
import time
from ultralytics import YOLO

# 1. Önceden eğitilmiş YOLOv8n (Nano) modelini yükle
model = YOLO("yolov8n.pt")

# 2. Modeli Pascal VOC veri seti üzerinde eğit (Fine-tuning)
# Rapor için sabitlenmiş hiperparametreler: imgsz=640, batch=16, epochs=10, optimizer=AdamW, lr0=0.001
start_train = time.time()

train_results = model.train(
    data="voc_custom.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    project="voc_project",
    name="yolov8n_voc",
    verbose=True
)

end_train = time.time()
training_duration = end_train - start_train

# 3. Eğitilen en iyi modeli (best.pt) ayrılan TEST kümesi üzerinde değerlendir
best_model = YOLO("voc_project/yolov8n_voc/weights/best.pt")
test_metrics = best_model.val(data="voc_custom.yaml", split="test")

# 4. Rapor için gerekli tüm metrikleri ve süreleri yazdır
print("\n" + "="*50)
print("📊 ÖDEV RAPORU İÇİN PERFORMANS ÖZETİ")
print("="*50)
print(f"Toplam Eğitim Süresi : {training_duration:.2f} saniye ({training_duration/60:.2f} dakika)")
print(f"Tahmin Süresi (Inference): {test_metrics.speed['inference']:.2f} ms / görüntü")
print(f"Precision (Kesinlik) : {test_metrics.box.mp:.4f}")
print(f"Recall (Duyarlılık)  : {test_metrics.box.mr:.4f}")
print(f"mAP@0.50 (IoU=0.50)  : {test_metrics.box.map50:.4f}")
print(f"mAP@0.50:0.95        : {test_metrics.box.map:.4f}")
print("="*50)

Ultralytics 8.4.212 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=voc_custom.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_voc, nbs=64, nms=None, opset=None, optimize=False, optimizer=Ada

FileNotFoundError: [Errno 2] No such file or directory: 'voc_project/yolov8n_voc/weights/best.pt'

In [6]:
from ultralytics import YOLO

# Doğru tam yoldan eğitilmiş en iyi modeli yükle
best_model = YOLO("/content/runs/detect/voc_project/yolov8n_voc/weights/best.pt")

# Ayrılan TEST kümesi üzerinde değerlendirme yap
test_metrics = best_model.val(data="voc_custom.yaml", split="test")

print("\n" + "="*55)
print("📊 ÖDEV RAPORU İÇİN TEST PERFORMANS ÖZETİ")
print("="*55)
print("Toplam Eğitim Süresi      : 2332.80 saniye (38.88 dakika / 0.648 saat)")
print(f"Tahmin Süresi (Inference) : {test_metrics.speed['inference']:.2f} ms / görüntü")
print(f"Precision (Kesinlik)      : {test_metrics.box.mp:.4f}")
print(f"Recall (Duyarlılık)       : {test_metrics.box.mr:.4f}")
print(f"mAP@0.50 (IoU=0.50)       : {test_metrics.box.map50:.4f}")
print(f"mAP@0.50:0.95             : {test_metrics.box.map:.4f}")
print("="*55)

Ultralytics 8.4.212 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,137,324 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1156.3±351.1 MB/s, size: 74.2 KB)
val: Scanning /content/datasets/VOC/labels/test_custom.cache... 2476 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2476/2476 798.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 155/155 7.0it/s 22.0s
                   all       2476       6210      0.768      0.711      0.786      0.571
             aeroplane        118        156      0.945      0.767      0.879       0.62
               bicycle        125        169      0.896      0.815      0.923      0.693
                  bird        135        217      0.835        0.7      0.801      0.548
                  boat         96        155      0.744      0.561      0.689      0.437
                bottle        

## Arayüz

In [10]:
import gradio as gr
from ultralytics import YOLO
from PIL import Image

# Eğitilen en iyi modeli yükle
model = YOLO("/content/runs/detect/voc_project/yolov8n_voc/weights/best.pt")

def nesne_tespiti(gorsel, guven_esigi):
    if gorsel is None:
        return None, "Lütfen bir fotoğraf yükleyin."

    sonuclar = model.predict(source=gorsel, conf=guven_esigi)
    cizimli_gorsel = sonuclar[0].plot()[:, :, ::-1]

    tespitler = []
    for box in sonuclar[0].boxes:
        sinif_id = int(box.cls[0])
        sinif_adi = model.names[sinif_id]
        skor = float(box.conf[0])
        tespitler.append(f"{sinif_adi}: %{skor*100:.1f}")

    ozet = "\n".join(tespitler) if tespitler else "Belirlenen eşik değerinde nesne tespit edilemedi."
    return Image.fromarray(cizimli_gorsel), ozet

arayuz = gr.Interface(
    fn=nesne_tespiti,
    inputs=[
        gr.Image(type="pil", label="Telefonla Çekilen Fotoğraf"),
        gr.Slider(minimum=0.10, maximum=0.90, value=0.35, step=0.05, label="Güven Eşiği (Confidence)")
    ],
    outputs=[
        gr.Image(type="pil", label="Model Tespit Çıktısı"),
        gr.Textbox(label="Tespit Edilen Sınıflar ve Güven Skorları")
    ],
    title="YOLOv8n - Pascal VOC Nesne Tespiti Uygulaması",
    description="Fotoğraf yükleyerek modelin nesne tespiti ve sınıflandırma başarımını test edebilirsiniz."
)

arayuz.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4b2ed917604d26020c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


##Çıktıları Zip Olarak İndirme

In [11]:
import os
import shutil
from google.colab import files

# GitHub reposu için temiz bir klasör oluştur
repo_dir = "/content/yolov8-pascal-voc-detection"
os.makedirs(f"{repo_dir}/results", exist_ok=True)

# 1. Eğitilen en iyi model ağırlığını kopyala
shutil.copy("/content/runs/detect/voc_project/yolov8n_voc/weights/best.pt", f"{repo_dir}/best.pt")

# 2. Rapor için önemli eğitim grafiklerini kopyala
grafikler = ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]
for g in grafikler:
    src = f"/content/runs/detect/voc_project/yolov8n_voc/{g}"
    if os.path.exists(src):
        shutil.copy(src, f"{repo_dir}/results/{g}")

# 3. Yerel ortamda çalışabilir Gradio uygulama dosyasını (app.py) oluştur
app_code = '''import gradio as gr
from ultralytics import YOLO
from PIL import Image

model = YOLO("best.pt")

def nesne_tespiti(gorsel, guven_esigi):
    if gorsel is None:
        return None, "Lütfen bir fotoğraf yükleyin."
    sonuclar = model.predict(source=gorsel, conf=guven_esigi)
    cizimli_gorsel = sonuclar[0].plot()[:, :, ::-1]

    tespitler = []
    for box in sonuclar[0].boxes:
        sinif_id = int(box.cls[0])
        sinif_adi = model.names[sinif_id]
        skor = float(box.conf[0])
        tespitler.append(f"{sinif_adi}: %{skor*100:.1f}")

    ozet = "\\n".join(tespitler) if tespitler else "Belirlenen eşik değerinde nesne tespit edilemedi."
    return Image.fromarray(cizimli_gorsel), ozet

arayuz = gr.Interface(
    fn=nesne_tespiti,
    inputs=[
        gr.Image(type="pil", label="Fotoğraf Yükle"),
        gr.Slider(minimum=0.10, maximum=0.90, value=0.35, step=0.05, label="Güven Eşiği (Confidence)")
    ],
    outputs=[
        gr.Image(type="pil", label="Model Tespit Çıktısı"),
        gr.Textbox(label="Tespit Edilen Sınıflar ve Güven Skorları")
    ],
    title="YOLOv8n - Pascal VOC Nesne Tespiti Uygulaması",
    description="Pascal VOC (20 Sınıf) veri seti ile eğitilmiş YOLOv8n modeli kullanılarak geliştirilen nesne tespiti arayüzü."
)

if __name__ == "__main__":
    arayuz.launch()
'''

with open(f"{repo_dir}/app.py", "w", encoding="utf-8") as f:
    f.write(app_code.strip())

# 4. requirements.txt oluştur
with open(f"{repo_dir}/requirements.txt", "w", encoding="utf-8") as f:
    f.write("ultralytics>=8.0.0\ngradio>=4.0.0\npillow>=10.0.0\n")

# 5. Klasörü ZIP yap ve indir
shutil.make_archive("/content/yolov8-pascal-voc-detection", "zip", repo_dir)
files.download("/content/yolov8-pascal-voc-detection.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>